In [1]:
%cd /workspace
import sys
import os

# Set PYTHONPATH environment variable for the kernel
robofin_path = os.path.join(os.getcwd(), 'robofin')
current_pythonpath = os.environ.get('PYTHONPATH', '')
if robofin_path not in current_pythonpath:
    os.environ['PYTHONPATH'] = f"{robofin_path}:{current_pythonpath}" if current_pythonpath else robofin_path

# Also add to sys.path for immediate effect
if robofin_path not in sys.path:
    sys.path.insert(0, robofin_path)

/workspace


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
def get_type(obj):
    """
    Function for displaying nested types.
    
    e.g. get_type(dict_str_float) -> "dict[str, float]"
    """
    if isinstance(obj, dict):
        if not obj:
            return "dict[?, ?]"
        key_types = {get_type(k) for k in obj.keys()}
        value_types = {get_type(v) for v in obj.values()}
        return f"dict[{', '.join(key_types)}, {', '.join(value_types)}]"
    elif isinstance(obj, list):
        if not obj:
            return "list[?]"
        elem_types = {get_type(elem) for elem in obj}
        return f"list[{', '.join(elem_types)}]"
    elif isinstance(obj, tuple):
        if not obj:
            return "tuple[?]"
        elem_types = [get_type(elem) for elem in obj]
        return f"tuple[{', '.join(elem_types)}]"
    elif isinstance(obj, set):
        if not obj:
            return "set[?]"
        elem_types = {get_type(elem) for elem in obj}
        return f"set[{', '.join(elem_types)}]"
    else:
        return type(obj).__name__

In [3]:
import numpy as np
import torch

def convert_to_numpy_f32(arr: np.ndarray | torch.Tensor) -> np.ndarray:
    """
    Convert a NumPy array or Torch tensor to a NumPy float32 array.
    
    Parameters
    ----------
    arr : np.ndarray or torch.Tensor
        Input array to convert.
    
    Returns
    -------
    np.ndarray
        Converted array with dtype float32.
    """
    if isinstance(arr, torch.Tensor):
        np_arr: np.ndarray = arr.cpu().numpy()
    elif isinstance(arr, np.ndarray):
        np_arr: np.ndarray = arr
    else:
        raise TypeError("convert_to_numpy_f32: Input must be a NumPy array or Torch tensor")
    return np_arr.astype(np.float32) 

In [4]:
%reload_ext autoreload
%autoreload 2

In [5]:
from robofin.robots import Robot


urdf_path = "assets/panda/panda_spheres.urdf"
robot_dof = 7
data_dir = "/workspace/datasets/ae_aristotle1_5mm_cubbies"
checkpoint_path = "/workspace/checkpoints/mpiformer_cubby.ckpt"
# urdf_path = "assets/gp7/gp7_spheres.urdf"
# robot_dof = 6
# data_dir = "/workspace/datasets/gp7_cubby_data"
# checkpoint_path = "/workspace/checkpoints/xuzop0ys-gp7/best-fabric-epoch13-step1000000.ckpt"

# Load the Robot class with the standard URDF file (that uses relative mesh filepaths)
robot = Robot(urdf_path)

## Dataloader/Replay buffer testing

In [6]:
training_model_parameters = {
  "robot_dof": robot_dof,  # robot's degrees of freedom
  "point_match_loss_weight": 1.0,  # point match loss (BC loss) weight for actor
  "actor_loss_weight": 1.0,        # RL actor loss weight
  "collision_loss_weight": 1.0,    # collision loss weight (only used for validation right now)
  "collision_loss_margin": 0.03,   # margin for collision loss [m]
  "min_lr": 1.0e-5,
  "max_lr": 5.0e-5,
  "warmup_steps": 5000,   # number of steps to warm up the learning rate linearly from min_lr to max_lr
  "weight_decay": 1e-4,   # weight decay for AdamW optimizers
  "gamma": 0.99,          # discount factor for Q-learning
  "tau": 0.005,           # target network's soft update rate
  "exploration_noise": 0.015,   # exploration noise for actor [std deviation]
  "action_clip": 0.5,         # clip the actor actions to be within [-action_clip, action_clip], can be "None"
  "target_actor_noise": 0.03,  # target actor noise for actor [std deviation] [TD3]
  "target_actor_noise_clip": 0.1, # target actor noise clip for actor [TD3]
  "use_huber_loss": True,     # use Huber loss for critic loss instead of MSE loss
  "grad_clip_norm": 1.0,      # gradient clipping norm
  "pc_bounds": [[-1.5, -1.5, -0.1], [1.5, 1.5, 1.5]],
  "rollout_length": 69,       # number of steps to rollout for the actor
}
shared_parameters = {
    "urdf_path": urdf_path,
    "num_robot_points": 2048,
    "goal_reward": 1.0,      # reward for reaching the goal
    "collision_reward": -1.0, # reward for colliding with an obstacle
    "step_reward": -0.02,        # reward for each step that doesn't terminate the episode
    "reward_scale": 1.0       # proportional scaling that applies to all rewards
}
config = {
    "num_workers": 8,                       # number of CPU workers for the data loaders
    "checkpoint_interval": 30,              # how often to save model checkpoints [minutes]
    "pretraining_steps": 200000,             # how many steps to pretrain for before starting to sample from actor's replay buffer [global steps]
    "start_using_actor_loss": 20000,         # number of steps to wait before actor loss activates
    "validate_every_n_steps": 10000,         # how often to run mid-epoch validation epochs [global steps]
    "collect_rollouts_every_n_steps": 50, # how often to collect actor rollouts for the replay buffer [global steps]
    "log_every_n_steps": 50,               # how often to log metrics to wandb [global steps]
    "replay_buffer_capacity": 100000,      # capacity of the replay buffer [number of transitions]
    "n_gpus": 1,                            # number of GPUs to use for training
    "train_batch_size": 12,                 # batch size for training
    "val_batch_size": 12,                   # batch size for validation
    "mid_epoch_max_val_batches": 1000,       # maximum number of batches to validate on during mid-epoch validation
    "mid_epoch_max_val_rollouts": 200,       # maximum number of rollouts to validate on during mid-epoch validation
    "end_epoch_max_val_batches": 2000,      # maximum number of batches to validate on at end-of-epoch validation
    "end_epoch_max_val_rollouts": 400,       # maximum number of rollouts to validate on at end-of-epoch validation
    "expert_fraction": 0.25,                # fraction of samples to sample from expert loader during fine-tuning
    "actor_delay": 2,                       # delay for actor network's and target network's updates [TD3]
    "max_epochs": 100,                      # maximum number of epochs to train for
    "mintest": False,                       # true: run minimal test run for debugging
    "logging": True,                        # true: log to wandb
    "load_model_from_checkpoint": True,
    "load_actor_only": True,
    "load_checkpoint_path": checkpoint_path,
    "save_checkpoint_dir": "/workspace/checkpoints"
}


In [7]:
from avoid_everything_except_exploration.data_loader import DataModule
data_module_parameters = {
    "data_dir": data_dir,
    "train_trajectory_key": "global_solutions",
    "val_trajectory_key": "global_solutions",
    "num_obstacle_points": 4096,
    "num_target_points": 128,
    "random_scale": 0.015,
    "include_reward": True,
    "shuffle": False,
}
dm = DataModule(
    train_batch_size=config["train_batch_size"],
    val_batch_size=config["val_batch_size"],
    num_workers=config["num_workers"],
    **data_module_parameters,
    **shared_parameters,
)
dm.setup("fit")
dl = dm.train_dataloader()

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded StateRewardDataset for training
Loaded StateRewardDataset for validation
Loaded TrajectoryDataset for validation
Loaded TrajectoryDataset for mini_train


In [8]:
from termcolor import cprint
from lightning.fabric import Fabric

from avoid_everything_except_exploration.replay import ReplayBuffer
from avoid_everything_except_exploration.col import CoLMotionPolicyTrainer
from avoid_everything_except_exploration.mixed_batch_provider import MixedBatchProvider


torch.set_float32_matmul_precision("high")

fabric = Fabric(accelerator="gpu", devices=1, precision="32-true")
fabric.launch()

expert_loader = fabric.setup_dataloaders(dm.train_dataloader(), move_to_device=True)
val_state_loader = fabric.setup_dataloaders(
    dm.val_state_dataloader(), move_to_device=True)
val_trajectory_loader = fabric.setup_dataloaders(
    dm.val_trajectory_dataloader(), move_to_device=True)

replay_buffer = ReplayBuffer(
    capacity=config["replay_buffer_capacity"],
    urdf_path=shared_parameters["urdf_path"],
    robot_dof=training_model_parameters["robot_dof"],
    num_robot_points=shared_parameters["num_robot_points"],
    num_target_points=data_module_parameters["num_target_points"],
    dataset=dm.data_train
)
mixed_provider = MixedBatchProvider(
    expert_loader=expert_loader, actor_replay=replay_buffer, async_prefetch=5
)
if config["load_model_from_checkpoint"]:
    cprint(f"Loading model from checkpoint {config['load_checkpoint_path']}", "blue")
    trainer = CoLMotionPolicyTrainer.load_from_checkpoint(
        config["load_checkpoint_path"],
        **(shared_parameters or {}),
        **(training_model_parameters or {}),
    )
else:
    cprint("Initializing new model", "red")
    trainer = CoLMotionPolicyTrainer(
        **(shared_parameters or {}),
        **(training_model_parameters or {}),
    )

trainer.configure_optimizers()
trainer.setup(fabric)

/usr/local/lib/python3.10/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Loading model from checkpoint /workspace/checkpoints/mpiformer_cubby.ckpt


In [9]:
batch, data_loader_iterations = mixed_provider.sample(
    config["train_batch_size"],
    expert_fraction=config["expert_fraction"],
    pretraining=True,
)
# for i in range(10):
#     trainer.actor_rollout(batch, replay_buffer)
#     viz_client.clear_ghost_robots()
#     print(i)

In [10]:
q = batch["configuration"]
pc_labels = batch["point_cloud_labels"]
pc = batch["point_cloud"]
qdeltas = trainer.actor(pc_labels, pc, q, trainer.pc_bounds)  # [B, 1, DOF] or [B, DOF]
a_raw = (qdeltas[:, -1, :] if qdeltas.dim()==3 else qdeltas)
a = trainer._apply_noise_in_joint_space(
    a_raw,
    noise_std=trainer.exploration_noise,
)

if trainer.action_clip is not None:
    a = torch.clamp(a, -trainer.action_clip, trainer.action_clip)

# next configuration
q_next = (q + a).clamp(-1, 1)
q_next_unn = robot.unnormalize_joints(q_next)

# next configuration without action noise
q_next_raw = (q + a_raw).clamp(-1, 1)
q_next_raw_unn = robot.unnormalize_joints(q_next_raw)

In [38]:
import viz_client

urdf_path = shared_parameters["urdf_path"]
if not os.path.exists(urdf_path):
    print(f"❌ URDF not found at {urdf_path}")
    raise FileNotFoundError(f"URDF not found at {urdf_path}")
viz_client.connect(urdf_path)

viz_server not running — starting…
Connected to viz_server


In [17]:
from avoid_everything_except_exploration.utils import visualization
import time

offset = 0
for i in range(50):
    visualization.visualize_problem(robot, sample=dl.dataset[offset + i])
    time.sleep(0.2)

In [47]:
from avoid_everything_except_exploration.utils import visualization
start_index = 16
end_index = 61
n_waypoints = end_index - start_index
sample = dl.dataset[start_index]
visualization.visualize_problem(robot, sample=sample)
# viz_client.clear_ghost_end_effector()
viz_client.clear_ghost_robots()
# viz_client.publish_config(robot.unnormalize_joints(sample["configuration"]))

for i in range(start_index+1, end_index):
    sample = dl.dataset[i]
    red = 1.0 - ((i - start_index) / (n_waypoints+30))
    green = ((i - start_index) / (n_waypoints))
    color = [red, green, 0.1]
    viz_client.publish_ghost_robot(robot.unnormalize_joints(sample["configuration"]), color=color, index=i)
    time.sleep(0.2)


In [ ]:
viz_client.shutdown()

In [ ]:
viz_client.publish_config(robot.neutral_config)

In [ ]:
from avoid_everything_except_exploration.utils import visualization
viz_client.clear_ghost_robots()
index = 5
visualization.visualize_problem(robot, sample={k: v[index] for k, v in batch.items()})

# visualize next configuration as green ghost robot
# viz_client.publish_ghost_robot(q_next_unn[index], color=[0.1, 0.9, 0.1], index=0)

# visualize next configuration without action noise as red ghost robot
# viz_client.publish_ghost_robot(q_next_raw_unn[index], color=[0.9, 0.1, 0.1], index=1)


In [ ]:
viz_client.clear_ghost_end_effector()

---

In [ ]:
from avoid_everything_except_exploration.utils import visualization
viz_client.clear_ghost_robots()
index = 2
visualization.visualize_problem(robot, sample={k: v[index] for k, v in batch.items()})
trainer.rollout_value_visualization(batch, index=index)

In [ ]:
viz_client.clear_ghost_robots()
viz_client.clear_ghost_end_effector()
viz_client.clear_obstacles()

In [ ]:
metrics = trainer.actor_rollout(batch, replay_buffer)
print(f"Mean episode reward: {metrics['avg_episode_reward']}")
print(f"Transitions collected: {metrics['transitions_collected']}")